1 Normalized column name and map all category into tkpi column

In [13]:
import pandas as pd
import numpy as np

# ============================================================
# 1. Read TKPI data
# ============================================================

input_file = "tkpi_raw.csv"
output_file = "tkpi_with_category.csv"

df = pd.read_csv("../raw/"+input_file)

print(df.head())



    KODE                           NAMA BAHAN      SUMBER AIR (g)  \
0  AR001                 Beras giling, mentah  KZGMI-2001    12.0   
1  AR002      Beras giling var pelita, mentah  KZGPI-1990    11.4   
2  AR003    Beras giling var rojolele, mentah  KZGPI-1990    12.0   
3  AR004                  Beras hitam, mentah  KZGMI-2001    12.9   
4  AR005  Beras jagung kuning, kering, mentah  KZGMI-2001    10.8   

   ENERGI (Kal) PROTEIN (g) LEMAK (g) KH (g) SERAT (g) ABU (g)  ...  \
0           357         8.4       1.7   77.1       0.2     0.8  ...   
1           369         9.5       1.4   77.1       0.4     0.6  ...   
2           357         8.4       1.7   77.1       0.2     0.8  ...   
3           351         8.0       1.3   76.9      20.1     0.9  ...   
4           358         5.5       0.1   82.7      10.0     0.9  ...   

  TEMBAGA (mg) SENG (mg) RETINOL (mcg) B-KAR (mcg) KAR-TOTAL (mcg)  \
0         0.10       0.5             0           0               0   
1         0.00    

Normalize column name

In [14]:
def normalize_column_name(col):
    col = str(col).strip()
    col = col.lower()

    # Replace Indonesian nutrient symbols / terms
    replacements = {
        "kode": "food_code",
        "nama bahan": "food_name",
        "sumber": "source",

        "air (g)": "water_g_100g",
        "energi (kal)": "energy_kcal_100g",
        "protein (g)": "protein_g_100g",
        "lemak (g)": "fat_g_100g",
        "kh (g)": "carb_g_100g",
        "serat (g)": "fiber_g_100g",
        "abu (g)": "ash_g_100g",

        "kalsium (mg)": "calcium_mg_100g",
        "fosfor (mg)": "phosphorus_mg_100g",
        "besi (mg)": "iron_mg_100g",
        "natrium (mg)": "sodium_mg_100g",
        "kalium (mg)": "potassium_mg_100g",
        "tembaga (mg)": "copper_mg_100g",
        "seng (mg)": "zinc_mg_100g",

        "retinol (mcg)": "retinol_mcg_100g",
        "b-kar (mcg)": "beta_carotene_mcg_100g",
        "kar-total (mcg)": "carotene_total_mcg_100g",

        "thiamin (mg)": "thiamin_mg_100g",
        "riboflavin (mg)": "riboflavin_mg_100g",
        "niasin (mg)": "niacin_mg_100g",
        "vit c (mg)": "vitamin_c_mg_100g",

        "bdd (%)": "bdd_percent"
    }

    if col in replacements:
        return replacements[col]

    # Fallback cleaning for unknown columns
    col = re.sub(r"\((.*?)\)", r"_\1", col)
    col = col.replace("%", "percent")
    col = col.replace("-", "_")
    col = col.replace("/", "_")
    col = re.sub(r"[^a-z0-9_]+", "_", col)
    col = re.sub(r"_+", "_", col)
    col = col.strip("_")

    return col


df.columns = [normalize_column_name(col) for col in df.columns]


# ============================================================
# 3. Check result
# ============================================================

print(df.columns.tolist())

print(df.head())

['food_code', 'food_name', 'source', 'water_g_100g', 'energy_kcal_100g', 'protein_g_100g', 'fat_g_100g', 'carb_g_100g', 'fiber_g_100g', 'ash_g_100g', 'calcium_mg_100g', 'phosphorus_mg_100g', 'iron_mg_100g', 'sodium_mg_100g', 'potassium_mg_100g', 'copper_mg_100g', 'zinc_mg_100g', 'retinol_mcg_100g', 'beta_carotene_mcg_100g', 'carotene_total_mcg_100g', 'thiamin_mg_100g', 'riboflavin_mg_100g', 'niacin_mg_100g', 'vitamin_c_mg_100g', 'bdd_percent']
  food_code                            food_name      source water_g_100g  \
0     AR001                 Beras giling, mentah  KZGMI-2001         12.0   
1     AR002      Beras giling var pelita, mentah  KZGPI-1990         11.4   
2     AR003    Beras giling var rojolele, mentah  KZGPI-1990         12.0   
3     AR004                  Beras hitam, mentah  KZGMI-2001         12.9   
4     AR005  Beras jagung kuning, kering, mentah  KZGMI-2001         10.8   

   energy_kcal_100g protein_g_100g fat_g_100g carb_g_100g fiber_g_100g  \
0              

Add TKPI data with new column category "food_category" to add 8 types of food

In [15]:
import pandas as pd
import numpy as np

# Make sure food_code and food_name are clean
df["food_code"] = df["food_code"].astype(str).str.strip().str.upper()
df["food_name"] = df["food_name"].astype(str).str.strip()

# ============================================================
# 1. Extract TKPI main group code
# Example:
# AR001 -> A
# DR008 -> D
# GR084 -> G
# ============================================================

df["tkpi_main_group"] = df["food_code"].str.extract(r"^([A-Z])")

# Optional: keep two-letter prefix for documentation
df["tkpi_group_code"] = df["food_code"].str.extract(r"^([A-Z]{2})")


# ============================================================
# 2. Assign category using food name keywords first
# Important:
# Keyword rules should come first because sugar/oil may not be
# easily identified only from TKPI code.
# ============================================================

def assign_category(row):
    code_group = row["tkpi_main_group"]
    name = str(row["food_name"]).lower()

    # Sugar / sweetener keywords
    sugar_keywords = [
        "gula", "madu", "sirup", "selai", "jam"
    ]

    # Oil / fat keywords
    oil_keywords = [
        "minyak", "lemak", "margarin", "mentega"
    ]

    if any(keyword in name for keyword in sugar_keywords):
        return "G", "Gula", "keyword_rule"

    if any(keyword in name for keyword in oil_keywords):
        return "M", "Minyak / Lemak", "keyword_rule"

    # Main TKPI group mapping based on first letter
    main_group_to_category = {
        "A": ("MP", "Makanan Pokok"),      # serealia / beras / jagung / tepung / nasi
        "B": ("MP", "Makanan Pokok"),      # umbi-umbian
        "C": ("LN", "Lauk Nabati"),        # kacang-kacangan, tahu, tempe
        "D": ("S", "Sayur"),              # sayuran
        "E": ("B", "Buah"),               # buah-buahan
        "F": ("LH", "Lauk Hewani"),       # daging
        "G": ("LH", "Lauk Hewani"),       # ikan / seafood
        "H": ("LH", "Lauk Hewani"),       # telur
        "J": ("SS", "Susu"),              # susu
    }

    if code_group in main_group_to_category:
        category_code, category_name = main_group_to_category[code_group]
        return category_code, category_name, "tkpi_main_group"

    return np.nan, np.nan, "unmapped"


df[["category_code", "category_name", "category_source"]] = df.apply(
    assign_category,
    axis=1,
    result_type="expand"
)


# ============================================================
# 3. Check result
# ============================================================

print("Total TKPI foods:", len(df))

print("\nTKPI main group summary:")
print(df["tkpi_main_group"].value_counts(dropna=False).sort_index())

print("\nTKPI two-letter prefix summary:")
print(df["tkpi_group_code"].value_counts(dropna=False).sort_index())

print("\nSimplified category summary:")
print(df["category_code"].value_counts(dropna=False))

print("\nCategory source summary:")
print(df["category_source"].value_counts(dropna=False))

unmapped = df[df["category_code"].isna()]

print("\nUnmapped rows:", len(unmapped))

if len(unmapped) > 0:
    print(unmapped[["food_code", "food_name", "tkpi_main_group", "tkpi_group_code"]].head(30))


# ============================================================
# 4. Save result
# ============================================================


print(f"\nSaved to: {output_file}")

Total TKPI foods: 1145

TKPI main group summary:
tkpi_main_group
A    135
B    109
C    138
D    227
E    127
F    122
G    179
H     18
J     17
K     18
M     17
N     37
Q      1
Name: count, dtype: int64

TKPI two-letter prefix summary:
tkpi_group_code
AP    114
AR     21
BP     76
BR     33
CP     86
CR     52
DP     61
DR    166
EP     15
ER    112
FP     88
FR     34
GP     95
GR     84
HP      7
HR     11
JP     11
JR      6
KP      4
KR     14
MP     17
NP     13
NR     24
QR      1
Name: count, dtype: int64

Simplified category summary:
category_code
LH     310
MP     241
S      216
LN     135
B      121
NaN     52
G       34
M       19
SS      17
Name: count, dtype: int64

Category source summary:
category_source
tkpi_main_group    1040
keyword_rule         53
unmapped             52
Name: count, dtype: int64

Unmapped rows: 52
     food_code                           food_name tkpi_main_group  \
1072     KR001  Kelapa setengah tua, daging, segar               K   
1073     

In [16]:
print(df.category_code.value_counts(dropna=False).sort_index())

category_code
B      121
G       34
LH     310
LN     135
M       19
MP     241
S      216
SS      17
NaN     52
Name: count, dtype: int64


In [17]:
import pandas as pd
import re
import unicodedata

def basic_clean_name(name):
    """
    Basic cleaning:
    - lowercase
    - remove extra spaces
    - normalize unicode
    - standardize punctuation spacing
    """
    if pd.isna(name):
        return ""

    name = str(name)
    name = unicodedata.normalize("NFKC", name)
    name = name.lower().strip()
    name = re.sub(r"\s+", " ", name)
    name = name.replace(" ,", ",")
    name = name.replace(", ", ", ")
    return name


def normalize_tkpi_name_for_matching(name):
    """
    Conservative normalization for matching TKPI food names to URT names.
    This removes general preparation descriptors but keeps identity-changing words.
    """
    name = basic_clean_name(name)

    # Remove common descriptors that usually describe physical state/preparation.
    # Be conservative: do NOT remove 'asin', 'asap', 'dendeng', 'sarden', etc.
    removable_phrases = [
        ", segar",
        ", mentah",
        ", rebus",
        ", kukus",
        ", masakan",
        ", matang",
        ", bakar",
    ]

    for phrase in removable_phrases:
        name = name.replace(phrase, "")

    # Some words are safe to remove only if they appear at the end
    # or after comma as descriptors.
    name = re.sub(r",\s*segar$", "", name)
    name = re.sub(r",\s*mentah$", "", name)
    name = re.sub(r",\s*rebus$", "", name)
    name = re.sub(r",\s*kukus$", "", name)
    name = re.sub(r",\s*masakan$", "", name)
    name = re.sub(r",\s*matang$", "", name)

    # Normalize separators
    name = name.replace("/", " ")
    name = re.sub(r"\s+", " ", name)
    name = name.strip(" ,.-")

    return name


# Example usage
df["food_name_clean"] = df["food_name"].apply(basic_clean_name)
df["food_name_match"] = df["food_name"].apply(normalize_tkpi_name_for_matching)

print(df[["food_name", "food_name_clean", "food_name_match"]].head(20))

                              food_name                      food_name_clean  \
0                  Beras giling, mentah                 beras giling, mentah   
1       Beras giling var pelita, mentah      beras giling var pelita, mentah   
2     Beras giling var rojolele, mentah    beras giling var rojolele, mentah   
3                   Beras hitam, mentah                  beras hitam, mentah   
4   Beras jagung kuning, kering, mentah  beras jagung kuning, kering, mentah   
5    Beras jagung putih, kering, mentah   beras jagung putih, kering, mentah   
6      Beras ketan hitam tumbuk, mentah     beras ketan hitam tumbuk, mentah   
7      Beras ketan putih tumbuk, mentah     beras ketan putih tumbuk, mentah   
8                  Beras ladang, mentah                 beras ladang, mentah   
9                   Beras menir, mentah                  beras menir, mentah   
10                      Beras parboiled                      beras parboiled   
11                 Beras tumbuk, mentah 

In [18]:
df.to_csv(output_file, index=False, encoding="utf-8-sig")